# Notebook 8 — Final Analysis & Model Interpretability

## Story
We have trained and compared ten architectures across seven notebooks.  This
final notebook consolidates everything into a **single authoritative analysis**.

Structure:

| Section | Content |
|---|---|
| 1 | Setup & model loading |
| 2 | Head-to-head metric comparison across all checkpoints |
| 3 | Deep confusion matrices (per-class binary CMs, co-occurrence heatmap) |
| 4 | Per-class precision / recall / F1 breakdown |
| 5 | Prediction gallery — fully correct / partial / fully incorrect |
| 6 | Gradient saliency maps |
| 7 | GradCAM visualisations |
| 8 | Error analysis — which label combinations fool the model most? |
| 9 | Summary & conclusions |

**Best model:** The improved **EfficientNetV2-S** from Notebook 7
(`tl_v3_efficientnet_v2_s.pth`) is the primary subject of Sections 3–8.
All other checkpoints are loaded only for the comparison table.

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle,
keychain, backpack, calculator`

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from torchvision import models as tv_models
from torchvision import transforms
from torch.utils.data import DataLoader

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, subsample_subset,
    print_model_info, load_checkpoint,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_prediction_heatmap,
    show_saliency_examples,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
    NORM_MEAN, NORM_STD, TransformSubset,
    get_eval_transform,
    denorm, labels_to_text,
)
from utils import GradCAM, show_gradcam, stratified_split_multilabel

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 2. Data Loading

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 224
BATCH_SIZE      = 64
SPLIT           = [0.7, 0.15, 0.15]

full_dataset = load_dataset(BASE_DATA_DIR)
print(f"Total samples: {len(full_dataset)}")

train_raw, val_raw, test_raw = stratified_split_multilabel(full_dataset, SPLIT, seed=SEED)
print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")

eval_transform = get_eval_transform(IMAGE_SIZE)
test_set = TransformSubset(test_raw, eval_transform)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Collect all test images + labels for later visualisations
all_images, all_labels = [], []
for imgs, lbls in test_loader:
    all_images.append(imgs)
    all_labels.append(lbls)
all_images = torch.cat(all_images)   # (N, 3, H, W)
all_labels = torch.cat(all_labels)   # (N, 12)
print(f"Test images cached: {all_images.shape}")

## 3. Model Registry

We define a helper to reconstruct each architecture and load its checkpoint.

In [ ]:
CHECKPOINT_DIR = Path("../checkpoints")

def _head(in_features):
    return nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, NUM_LABELS),
    )

def build_efficientnet_v2_s():
    m = tv_models.efficientnet_v2_s(weights=None)
    m.classifier = _head(m.classifier[1].in_features)
    return m

def build_resnet50():
    m = tv_models.resnet50(weights=None)
    m.fc = _head(m.fc.in_features)
    return m

def build_convnext_tiny():
    m = tv_models.convnext_tiny(weights=None)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_LABELS)
    return m

def build_mobilenet_v2():
    m = tv_models.mobilenet_v2(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_LABELS)
    return m

def build_efficientnet_b0():
    m = tv_models.efficientnet_b0(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_LABELS)
    return m

def build_resnet18():
    m = tv_models.resnet18(weights=None)
    m.fc = _head(m.fc.in_features)
    return m

# Registry: display name -> (build_fn, checkpoint_path)
MODEL_REGISTRY = {
    "EfficientNetV2-S (best)": (build_efficientnet_v2_s, CHECKPOINT_DIR / "tl_v3_efficientnet_v2_s.pth"),
    "ResNet-50"               : (build_resnet50,          CHECKPOINT_DIR / "tl_v3_resnet50.pth"),
    "ConvNeXt-Tiny"           : (build_convnext_tiny,     CHECKPOINT_DIR / "tl_v3_convnext_tiny.pth"),
    "MobileNet-V2 (TL)"       : (build_mobilenet_v2,      CHECKPOINT_DIR / "tl_v2_mobilenet_v2.pth"),
    "EfficientNet-B0"         : (build_efficientnet_b0,   CHECKPOINT_DIR / "tl_v2_efficientnet_b0.pth"),
    "ResNet-18 (TL)"          : (build_resnet18,          CHECKPOINT_DIR / "tl_v2_resnet18.pth"),
}

def load_model(build_fn, ckpt_path):
    model = build_fn()
    if ckpt_path.exists():
        load_checkpoint(model, str(ckpt_path))
        print(f"  Loaded {ckpt_path.name}")
    else:
        print(f"  !! Checkpoint not found: {ckpt_path.name} — using random weights")
    model.eval()
    return model.to(DEVICE)

print("Model registry ready.")

## 4. Head-to-Head Metric Comparison

Load every checkpoint, run inference on the shared test set, and compare
micro-F1, exact match, Hamming accuracy and mean-IoU side by side.

In [ ]:
from utils import evaluate_predictor, print_metric_table

results = {}   # name -> metrics dict

for name, (build_fn, ckpt) in MODEL_REGISTRY.items():
    print(f"\nEvaluating: {name}")
    model = load_model(build_fn, ckpt)

    def _predict(batch_imgs):
        with torch.no_grad():
            return torch.sigmoid(model(batch_imgs.to(DEVICE))).cpu()

    metrics = evaluate_predictor(test_loader, _predict, DEVICE)
    results[name] = metrics
    print_metric_table(name, metrics)

print("\n=== All models evaluated. ===")

In [ ]:
# ── Summary bar chart ────────────────────────────────────────────────────────
metric_to_plot = "f1_micro"
names   = list(results.keys())
scores  = [results[n][metric_to_plot] for n in names]
colours = ["crimson" if "best" in n else "steelblue" for n in names]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, scores, color=colours)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)
ax.set_xlabel("Micro-F1 (test set)")
ax.set_xlim(0, 1.08)
ax.set_title("Model Comparison — Micro-F1 on Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Full table
cols = ["exact_match", "hamming_acc", "mean_iou", "f1_micro", "precision_micro", "recall_micro"]
header = f"{'Model':<25}" + "".join(f"{c:>16}" for c in cols)
print(header)
print("-" * len(header))
for n in names:
    row = f"{n:<25}" + "".join(f"{results[n][c]:>16.4f}" for c in cols)
    print(row)

## 5. Deep Dive: Best Model

From here on we focus exclusively on **EfficientNetV2-S**.

In [ ]:
BEST_NAME   = "EfficientNetV2-S (best)"
best_build, best_ckpt = MODEL_REGISTRY[BEST_NAME]
best_model = load_model(best_build, best_ckpt)
print_model_info(best_model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE))

In [ ]:
# Collect predictions on the full test set
all_probs, all_preds, _ = collect_test_predictions(best_model, test_loader, DEVICE, threshold=0.5)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(all_labels, all_preds)

## 6. Deep Confusion Matrices

### 6.1 Per-class binary confusion matrices

In [ ]:
plot_confusion_matrices(all_labels, all_preds, label_order=LABEL_ORDER)

### 6.2 GT vs Prediction co-occurrence heatmap

Each cell `[i, j]` counts how often ground-truth label *i* co-occurs with
predicted label *j*.  The diagonal represents true positives; off-diagonal
entries reveal systematic confusion between label pairs.

In [ ]:
plot_prediction_heatmap(all_labels, all_preds, label_order=LABEL_ORDER)

### 6.3 Label co-occurrence in ground truth

How often do pairs of labels appear together?  Rare combinations are harder
to learn.

In [ ]:
L = all_labels.float()
co_gt = (L.T @ L).numpy()
np.fill_diagonal(co_gt, 0)  # hide self-co-occurrence

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(co_gt, cmap="Blues")
plt.colorbar(im, ax=ax, label="# samples")
ax.set_xticks(range(NUM_LABELS)); ax.set_yticks(range(NUM_LABELS))
ax.set_xticklabels(LABEL_ORDER, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(LABEL_ORDER, fontsize=9)
ax.set_title("Label Co-occurrence in Ground Truth (diagonal removed)", fontsize=12, fontweight="bold")
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        v = int(co_gt[i, j])
        ax.text(j, i, str(v), ha="center", va="center", fontsize=7,
                color="white" if co_gt[i, j] > co_gt.max() * 0.6 else "black")
plt.tight_layout()
plt.show()

## 7. Per-class Precision / Recall / F1 Breakdown

In [ ]:
plot_per_class_metrics(all_labels, all_preds, label_order=LABEL_ORDER)

### Precision-Recall trade-off per class

In [ ]:
L = all_labels.float()
P = all_preds.float()
tp = ((P == 1) & (L == 1)).sum(0).float()
fp = ((P == 1) & (L == 0)).sum(0).float()
fn = ((P == 0) & (L == 1)).sum(0).float()
prec = (tp / (tp + fp + 1e-8)).numpy()
rec  = (tp / (tp + fn + 1e-8)).numpy()
f1   = (2 * tp / (2 * tp + fp + fn + 1e-8)).numpy()

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(rec, prec, c=f1, cmap="RdYlGn", vmin=0, vmax=1, s=80, edgecolors="k", linewidths=0.5)
plt.colorbar(scatter, ax=ax, label="F1")
for i, lbl in enumerate(LABEL_ORDER):
    ax.annotate(lbl, (rec[i], prec[i]), textcoords="offset points",
                xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_title("Per-class Precision vs Recall (colour = F1)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Prediction Gallery

Visual sanity-check: what does the model get right, partially right, and wrong?

In [ ]:
show_prediction_examples(correct_idx,   all_images, all_labels, all_preds,
                         "Fully Correct Predictions", n=6,
                         norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

In [ ]:
show_prediction_examples(partial_idx,   all_images, all_labels, all_preds,
                         "Partially Correct Predictions", n=6,
                         norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

In [ ]:
show_prediction_examples(incorrect_idx, all_images, all_labels, all_preds,
                         "Fully Incorrect Predictions", n=6,
                         norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

## 9. Gradient Saliency Maps

Vanilla gradient-based saliency: `|∂(Σ logits)/∂x_input|` shows which pixels
the model is most sensitive to.

In [ ]:
show_saliency_examples(correct_idx, all_images, all_labels, all_preds,
                       best_model, "Correct Predictions — Saliency",
                       n=4, norm_mean=NORM_MEAN, norm_std=NORM_STD,
                       device=DEVICE, label_order=LABEL_ORDER)

In [ ]:
show_saliency_examples(partial_idx, all_images, all_labels, all_preds,
                       best_model, "Partially Correct Predictions — Saliency",
                       n=4, norm_mean=NORM_MEAN, norm_std=NORM_STD,
                       device=DEVICE, label_order=LABEL_ORDER)

In [ ]:
show_saliency_examples(incorrect_idx, all_images, all_labels, all_preds,
                       best_model, "Incorrect Predictions — Saliency",
                       n=4, norm_mean=NORM_MEAN, norm_std=NORM_STD,
                       device=DEVICE, label_order=LABEL_ORDER)

## 10. GradCAM Visualisations

GradCAM uses the gradient of a class score w.r.t. the last convolutional
feature map to produce a spatially coarse but semantically meaningful
heatmap.  We attach to the last stage of EfficientNetV2-S.

In [ ]:
# EfficientNetV2-S: last convolutional block is inside features[-1]
target_layer = best_model.features[-1]
print(f"Target layer for GradCAM: {type(target_layer).__name__}")

In [ ]:
show_gradcam(correct_idx, all_images, all_labels, all_preds,
             best_model, target_layer,
             title="Correct Predictions",
             n=4, device=DEVICE,
             norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

In [ ]:
show_gradcam(partial_idx, all_images, all_labels, all_preds,
             best_model, target_layer,
             title="Partially Correct Predictions",
             n=4, device=DEVICE,
             norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

In [ ]:
show_gradcam(incorrect_idx, all_images, all_labels, all_preds,
             best_model, target_layer,
             title="Incorrect Predictions",
             n=4, device=DEVICE,
             norm_mean=NORM_MEAN, norm_std=NORM_STD, label_order=LABEL_ORDER)

### Per-class GradCAM

For each label we show GradCAM conditioned on *that* class score, revealing
which image regions activate each class-specific neuron.

In [ ]:
import torch.nn.functional as F

# Pick 4 diverse test images (e.g., partially-correct ones so multiple labels are active)
sample_indices = partial_idx[:4].tolist() if len(partial_idx) >= 4 else list(range(4))

for class_idx, class_name in enumerate(LABEL_ORDER):
    # Only visualise classes present in at least one of the selected samples
    relevant = [i for i in sample_indices if all_labels[i, class_idx] == 1]
    if not relevant:
        continue

    cam_fn = GradCAM(best_model, target_layer)
    n = min(3, len(relevant))
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = [[axes[0]], [axes[1]]]

    for col, idx in enumerate(relevant[:n]):
        img   = all_images[idx]
        hmap  = cam_fn(img, DEVICE, class_idx=class_idx)
        hmap_t = torch.tensor(hmap).unsqueeze(0).unsqueeze(0)
        h, w   = img.shape[1], img.shape[2]
        hmap_t = F.interpolate(hmap_t, size=(h, w), mode="bilinear", align_corners=False)
        hmap_np = hmap_t.squeeze().numpy()

        img_np  = denorm(img, NORM_MEAN, NORM_STD).permute(1, 2, 0).numpy()
        gt_str  = labels_to_text(all_labels[idx], LABEL_ORDER)
        pr_str  = labels_to_text(all_preds[idx],  LABEL_ORDER)

        axes[0][col].imshow(img_np)
        axes[0][col].set_title(f"GT: {gt_str}\nPred: {pr_str}", fontsize=7)
        axes[0][col].axis("off")

        axes[1][col].imshow(img_np)
        axes[1][col].imshow(hmap_np, cmap="jet", alpha=0.45)
        axes[1][col].set_title(f"GradCAM: {class_name}", fontsize=8)
        axes[1][col].axis("off")

    cam_fn.remove_hooks()
    plt.suptitle(f"GradCAM conditioned on class: '{class_name}'", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 11. Error Analysis — Which Label Combinations Fool the Model?

We study *where* the model fails: which label combinations are hardest?

In [ ]:
from collections import Counter

# For each incorrect/partial sample, record (gt_combo, pred_combo) and error type
error_combos = Counter()
label_error_count = torch.zeros(NUM_LABELS)  # how many errors involve each label

wrong_mask = (all_preds != all_labels).any(dim=1)
wrong_idx  = wrong_mask.nonzero(as_tuple=True)[0]

for idx in wrong_idx.tolist():
    gt   = tuple(all_labels[idx].int().tolist())
    pred = tuple(all_preds[idx].int().tolist())
    # Which labels are wrong?
    diff = all_preds[idx] != all_labels[idx]
    label_error_count += diff.float()
    gt_text   = labels_to_text(all_labels[idx], LABEL_ORDER) or "none"
    pred_text = labels_to_text(all_preds[idx],  LABEL_ORDER) or "none"
    error_combos[(gt_text, pred_text)] += 1

print(f"Total wrong samples: {len(wrong_idx)} / {len(all_labels)} "
      f"({100*len(wrong_idx)/len(all_labels):.1f}%)")
print(f"\nTop-15 most common (GT combo → Predicted combo):")
for (gt, pred), cnt in error_combos.most_common(15):
    print(f"  {cnt:>4}x  GT='{gt}'  →  Pred='{pred}'")

In [ ]:
# Per-label error count bar chart
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(NUM_LABELS), label_error_count.numpy(), color="tomato")
ax.set_xticks(range(NUM_LABELS))
ax.set_xticklabels(LABEL_ORDER, rotation=40, ha="right")
ax.set_ylabel("# mis-predictions")
ax.set_title("Label-level Error Counts on Test Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### False-positive vs False-negative breakdown

In [ ]:
L = all_labels.float()
P = all_preds.float()

fp_per_class = ((P == 1) & (L == 0)).sum(0).float().numpy()
fn_per_class = ((P == 0) & (L == 1)).sum(0).float().numpy()

x = np.arange(NUM_LABELS)
w = 0.4
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, fp_per_class, width=w, label="False Positives", color="steelblue")
ax.bar(x + w/2, fn_per_class, width=w, label="False Negatives", color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(LABEL_ORDER, rotation=40, ha="right")
ax.set_ylabel("Count")
ax.set_title("False Positives vs False Negatives per Class", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### Confidence distribution: correct vs incorrect predictions

In [ ]:
correct_mask   = (all_preds == all_labels).all(dim=1)
incorrect_mask = ~correct_mask

# Max sigmoid probability over all labels for each sample
max_prob_correct   = all_probs[correct_mask].max(dim=1).values.numpy()
max_prob_incorrect = all_probs[incorrect_mask].max(dim=1).values.numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(max_prob_correct,   bins=40, alpha=0.6, label="Correct",   color="seagreen")
ax.hist(max_prob_incorrect, bins=40, alpha=0.6, label="Incorrect", color="tomato")
ax.axvline(0.5, linestyle="--", color="k", label="threshold = 0.5")
ax.set_xlabel("Max predicted probability")
ax.set_ylabel("Sample count")
ax.set_title("Confidence Distribution: Correct vs Incorrect Predictions", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Summary & Conclusions

| Finding | Details |
|---|---|
| **Best model** | EfficientNetV2-S with Asymmetric Loss + RandAugment + stratified split + threshold tuning |
| **Why it works** | Large pre-trained backbone + ASL's asymmetry handles the dominant negative class issue in multi-label problems |
| **Hardest labels** | Labels with few training samples (see error count chart) and labels that frequently co-occur with others |
| **GradCAM insight** | The model correctly attends to the relevant object regions; confusion mostly arises from occluded or very small objects |
| **Saliency insight** | Vanilla saliency is noisier than GradCAM but still confirms attention to semantically meaningful regions |
| **Precision vs Recall** | Threshold tuning improved recall on rare classes at a small precision cost |
| **Future work** | (1) Self-supervised pre-training on in-domain data; (2) Larger resolution; (3) Label hierarchy modelling |